# Subject Dependent Experiments
model trained on one individual subject's samples, shuffled and divided into an 80-20 train-test split

In [1]:
import numpy as np
import tensorflow as tf
from data_preprocess import extract_subj_dep
import hybrid_model_structures as models

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from keras.optimizers import Adam

#INPUT

#subjects directory path
#The data should be extracted in the same directory as this script.
dir_path_f = "../../Data/EF-WG/Raw/"
dir_path_e = "../../Data/EF-WG/Raw/"

#subject number 
subject="01"   

#result written to this file
result_file="Results/log.txt"

#######################
#Deep Learning settings:

baseline_model=False    #using baselines or EFNet
resnet=False
if baseline_model:
    lr=0.00005
    EPOCHS=70
else:
    lr=0.001
    EPOCHS=70
BATCH_SIZE = 32
RANDOM_STATE=38

X_eeg, X_fnirs, Y = extract_subj_dep(subject,baseline_model)

# Standardize our signals data
# Reshaping the data samples into 2D applying the scaling and then reshaping back
scaler = StandardScaler()
XeegS = scaler.fit_transform(X_eeg.reshape(-1, X_eeg.shape[-1])).reshape(X_eeg.shape)
XfnirsS = scaler.fit_transform(X_fnirs.reshape(-1, X_fnirs.shape[-1])).reshape(X_fnirs.shape)

#adjustment to labels to make compatible with binary_crossentropy
Y=Y[:,0]

print(XeegS.shape)
print(XfnirsS.shape)
print(Y.shape)

#reshaping data into correct form for input into models.
if baseline_model:
    XeegS_3D=XeegS
    XfnirsS_3D=XfnirsS
else:
    #reshape into 2D and 1 channel data (3D samples list) for EF-Net.
    XeegS_3D=np.reshape(XeegS,(XeegS.shape[0],XeegS.shape[1],XeegS.shape[2],1))
    XfnirsS_3D=np.reshape(XfnirsS,(XfnirsS.shape[0],XfnirsS.shape[1],XfnirsS.shape[2],1))
XeegS_3D.shape

#Split all samples into training and testing splits
Xeeg_train, Xeeg_test, y_train, y_test = train_test_split(XeegS_3D, Y, test_size=0.2,random_state=RANDOM_STATE)
print(Xeeg_train.shape,Xeeg_test.shape)

Xfnirs_train, Xfnirs_test, y_train2, y_test2 = train_test_split(XfnirsS_3D, Y, test_size=0.2, random_state=RANDOM_STATE)
print(Xfnirs_train.shape, Xfnirs_test.shape)

if baseline_model:
    hybridmodel= models.eeg_fnirs_vgg(Xeeg_train,Xfnirs_train, resnet)
else:
    hybridmodel=models.eeg_fnirs_cnn_v2()
hybridmodel.summary()

hybridmodel.compile(optimizer=Adam(learning_rate=lr),loss='binary_crossentropy',metrics=['accuracy','Precision','Recall','AUC'])

#save best model weights
checkpoint_filepath = '/tmp/checkpoint.weights.h5'
model_checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_filepath,
    save_weights_only=True,
    monitor='val_accuracy',
    mode='max',
    save_best_only=True,
    verbose=1)

hybridmodel.fit(x=[Xeeg_train, Xfnirs_train], y=y_train, validation_data=([Xeeg_test, Xfnirs_test], y_test), epochs=EPOCHS, callbacks=[model_checkpoint_callback])
hybridmodel.load_weights(checkpoint_filepath)
probabilities = hybridmodel.predict([Xeeg_test, Xfnirs_test])
y_preds= np.round(probabilities)

loss,accuracy,precision,recall,AUC = hybridmodel.evaluate([Xeeg_test,Xfnirs_test], y_test)

2025-05-08 13:44:27.461382: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-05-08 13:44:27.478656: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-05-08 13:44:27.484040: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-05-08 13:44:27.498246: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


tasks shape (100, 72)
Length of data samples,  6
tasks shape (100, 72)
Length of data samples,  12
tasks shape (100, 72)
Length of data samples,  18
tasks shape (100, 72)
Length of data samples,  24
tasks shape (100, 72)
Length of data samples,  30
tasks shape (100, 72)
Length of data samples,  36
tasks shape (100, 72)
Length of data samples,  42
tasks shape (100, 72)
Length of data samples,  48
tasks shape (100, 72)
Length of data samples,  54
tasks shape (100, 72)
Length of data samples,  60
tasks shape (100, 72)
Length of data samples,  66
tasks shape (100, 72)
Length of data samples,  72
tasks shape (100, 72)
Length of data samples,  78
tasks shape (100, 72)
Length of data samples,  84
tasks shape (100, 72)
Length of data samples,  90
tasks shape (100, 72)
Length of data samples,  96
tasks shape (100, 72)
Length of data samples,  102
tasks shape (100, 72)
Length of data samples,  108
tasks shape (100, 72)
Length of data samples,  114
tasks shape (100, 72)
Length of data samples,  1

2025-05-08 13:44:33.245695: I tensorflow/core/common_runtime/gpu/gpu_device.cc:2021] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 56679 MB memory:  -> device: 0, name: NVIDIA A800 80GB PCIe, pci bus id: 0000:a1:00.0, compute capability: 8.0


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 500, 30,   │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 494, 30,   │        256 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 488, 30,   │      7,200 │ conv2d[0][0]      │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 482, 30,   │      7,200 │ conv2d_1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 68, 30,    │          0 │ conv2d_2[0][0]    │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, 25, 72, 1) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 68, 30,    │          0 │ max_pooling2d[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_6 (Conv2D)   │ (None, 22, 72,    │        160 │ input_layer_1[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 68, 30,    │        128 │ dropout[0][0]     │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_7 (Conv2D)   │ (None, 19, 72,    │      4,128 │ conv2d_6[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 65, 27,    │     32,832 │ batch_normalizat… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 4, 72, 32) │          0 │ conv2d_7[0][0]    │
│ (MaxPooling2D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 62, 24,    │     65,600 │ conv2d_3[0][0]    │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 4, 72, 32) │          0 │ max_pooling2d_2[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_5 (Conv2D)   │ (None, 59, 21,    │     65,600 │ conv2d_4[0][0]    │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 4, 72, 32) │        128 │ dropout_3[0][0]   │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 14, 5, 64) │          0 │ conv2d_5[0][0]    │
│ (MaxPooling2D)      │                   │            │                 

...

Epoch 1/70


/data_main/junlin/conda_env/mbi_tf/lib/python3.10/site-packages/keras/src/models/functional.py:225: UserWarning: The structure of `inputs` doesn't match the expected structure: ['keras_tensor', 'keras_tensor_1']. Received: the structure of inputs=('*', '*')
  warnings.warn(
I0000 00:00:1746683078.946912  314065 service.cc:146] XLA service 0x70ccd4024640 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1746683078.946997  314065 service.cc:154]   StreamExecutor device (0): NVIDIA A800 80GB PCIe, Compute Capability 8.0
2025-05-08 13:44:39.191915: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2025-05-08 13:44:39.726347: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:531] Loaded cuDNN version 90501
Could not load symbol cuFuncGetName. Error: /lib/x86_64-linux-gnu/libcuda.so.1: undefined symbol: cuFuncGetName


5/9 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - AUC: 0.5600 - Precision: 0.5312 - Recall: 0.3800 - accuracy: 0.5171 - loss: 0.6904 

I0000 00:00:1746683089.595308  314065 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.



Epoch 1: val_accuracy improved from -inf to 0.59722, saving model to /tmp/checkpoint.weights.h5
9/9 ━━━━━━━━━━━━━━━━━━━━ 19s 653ms/step - AUC: 0.5735 - Precision: 0.5497 - Recall: 0.4373 - accuracy: 0.5355 - loss: 0.6895 - val_AUC: 0.6749 - val_Precision: 0.5600 - val_Recall: 0.8000 - val_accuracy: 0.5972 - val_loss: 0.6789
Epoch 2/70
7/9 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - AUC: 0.6243 - Precision: 0.5343 - Recall: 0.6605 - accuracy: 0.5631 - loss: 0.6822 
Epoch 2: val_accuracy did not improve from 0.59722
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - AUC: 0.6221 - Precision: 0.5386 - Recall: 0.6520 - accuracy: 0.5610 - loss: 0.6818 - val_AUC: 0.7054 - val_Precision: 0.5652 - val_Recall: 0.7429 - val_accuracy: 0.5972 - val_loss: 0.6653
Epoch 3/70
7/9 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - AUC: 0.7297 - Precision: 0.6209 - Recall: 0.7762 - accuracy: 0.6583 - loss: 0.6592 
Epoch 3: val_accuracy improved from 0.59722 to 0.65278, saving model to /tmp/checkpoint.weights.h5
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s